# 解答例（教員用）　第17回
***
> このノートブックは **模範解答の一例**です．コードの空欄（`___`）を埋め，探索は複数ラウンドに分けて実行し，解答欄には実際に得られた出力の数値を根拠に書いています．
> 乱数を使う箇所は固定していますが，ライブラリのバージョンや実行環境によって数値が **小数第2〜3位ほどずれる**ことがあります．探索型の問題なので，**別の探索経路・別の最良設定でも，根拠が筋の通っていれば正解**です．

# 第17回　総合課題 — 手書き数字認識
***
> **前提**: 第16回で学習・保存した `best_mnist_model.pth` を使い，自作の手書き数字で推論します。
>
> **実行環境**: 手書き推論 UI は **Google Colab** 上での実行を推奨します（`google.colab.output` を使用）。Colab では第14〜16回を順に実行してから本課題に取り組んでください。ローカル Jupyter では問題1と問題3（考察）のみ実施可能です。

> ⚠️ **この課題で身につけること：コーディングではなく「AI（ニューラルネット）の中身の理解」です。**
>
> コードは AI に書かせても構いません。重要なのは「**なぜその処理を選ぶのか**」「**パラメータや特徴量を変えると結果がどう変わるのか**」を理解し、提出物で示すことです。本回は **考察中心**です。各問には学習目標を示すタグが付いています。
>
> | タグ | 意味 | あなたがすること |
> |---|---|---|
> | **【骨格】** | 動くコードは与えられている | 設計上の決定点（数値・選択肢・特徴量）だけを変更する |
> | **【選択】** | 適切な手法を選ぶ問題 | 複数候補から選び、**理由**を解答用コードセルに書く |
> | **【実験】** | 試行錯誤の記録 | パラメータ等を変えて結果を表に記録し、**考察**する |
> | **【説明】** | 理解の証跡 | 与えられたコードの各行に `# 説明:` で意味を書く |
> | **【探索】** | 最適設定を自分で探す | **仮説 → 試行 → 記録 → 次の仮説** を何度も繰り返す。全試行が自動で履歴に残る |
> | **【指示書】** | 知識を LLM に渡す | 実験で得た知識を、LLM が使える「指示書」に書き起こし、LLM の出力を自分のログで採点する |
> | **【考察】** | 理解の証跡 | なぜそうなるのかを自分の言葉で説明する |
>
> コードは原則として完成形ですが、**核心となる最低限の数行は `# ★あなたが書く★` として空欄**にしてあります。要となる処理は自分で書けることも確認します（ボイラープレートは提供済み）。
>
> 各問の **✍️ 解答用コードセル**（`# (1-a)` 形式の変数・文字列）に、選んだ選択肢・理由・観察・考察を**項目ごとに**記入してください。これが採点対象です。

---

## 🧪 この回の進め方：最適な設定は「自分で」探す

この回では **「正解のパラメータ」は教えません**。答えは、あなたが何度も試して見つけます。

```
  ①仮説を立てる → ②試す（1回1行の記録が自動で残る） → ③結果を見て仮説を直す → ①へ戻る
```

### 試行錯誤の記録帳 `Lab`（最初のコードセルで用意済み。中身を読んで使い方を理解すること）

| 使い方 | 何をするか |
|---|---|
| `lab.df()` | これまでの全試行を表で見る（試行番号・設定・スコア・仮説） |
| `lab.plot(x="threshold", group=...)` | 設定とスコアの関係、ベスト更新の推移をグラフにする |
| `lab.report()` | 探索の自己診断（回数・範囲・端に張り付いていないか・仮説を書いたか） |
| `lab.final_test(...)` | **テストデータでの最終評価。1回しか実行できない** |
| `lab.save()` | 全履歴を CSV に保存する |

### ルール（ここが「AIの理解」の中心）

1. **探索に使ってよいのは検証スコア（現場入力を模した画像の「開発用」半分での正解率）だけ**。テストデータは最後に **1回だけ**（見てから設定を変えたらデータリーク）。
2. **試行のたびに「仮説」を書く**（何を確かめたい試行か）。やみくもな総当たりは評価されません。
3. **粗く → 細かく**：まず桁・範囲を広く振って良さそうな所を掴み、次にその周辺を細かく調べる。
4. **ベストが探索範囲の端に来たら、範囲を広げて確かめる**。
5. 上位の設定同士の差が、ばらつきより小さいなら「差がある」とは言えない。
   ※この回の「テスト」は、現場入力を模した画像の **もう半分（holdout）**。最後に 1回だけ使います。
6. **実験の前に予測を書く**。予測と結果のズレが、あなたが得た「知識」の証拠になる。
7. 最後に、得た知識を **LLM への指示書**にまとめる（最終問題）。

### 評価の観点

| 観点 | 見られること |
|---|---|
| 試行の量と質 | 目安回数以上・仮説が毎回ある・粗→細の流れがある |
| 結果の読み取り | 外れた仮説から何を学んだか（試行番号 `#n` を引用して書く） |
| 検証の作法 | テストは1回だけ／検証スコアとテストの差を正しく解釈している |
| 知識の言語化 | 自分の実験結果を、LLM が実行できる指示書に落とせている |

## 目次
1. モデルの読み込み
2. 手書き推論 UI（完成コード — 編集不要）
3. 推論結果の記録
4. 考察
5. 実験で得た知識を LLM への指示書にする【指示書】

---

## この回で学ぶこと

### 「学習データと推論データの分布ズレ」という根本的な問題

第14〜16回では MNIST データセットで学習し，MNIST のテストデータで評価してきた。しかし今回は「自分の手書き文字」という，学習データとは**異なる分布**のデータで推論する。

この問題を **Distribution Shift（分布シフト）** または **Domain Shift** と呼ぶ：

```
【学習時のデータ分布】         【推論時のデータ分布】
MNIST（白背景・黒文字・          自分の手書き（UI キャンバス：
  標準的なサイズ・               黒背景・白文字・
  中央揃え）                     個人差がある文字スタイル）
```

これが AI の実運用での最大の課題の一つだ。

### 前処理の一致がなぜ重要か

モデルは学習時の前処理を「前提」として学習している。推論時に異なる前処理をすると，モデルが意図しない入力を受け取ることになる：

```
学習時: PIL画像 → ToTensor() → [0,1]正規化 → 白地に黒文字
推論時（今回）: Canvas画像（黒地に白文字）→ 色反転 → リサイズ(28,28) → 同じ正規化
```

今回のコードで `arr = 1.0 - arr` （色反転）を行っているのは，この分布ズレを補正するためだ。

### 信頼度（Confidence）の意味

`torch.softmax(logits, dim=1)` の出力は各クラスの「確率」だ。最大値が「信頼度」として表示される。ただし：

- 信頼度が高い ≠ 必ず正解（モデルが確信を持って間違えることもある）
- これを「過信（Overconfidence）」と呼ぶ。深層学習モデルによく見られる問題だ
- 対策：Temperature Scaling, MC Dropout などの不確かさ推定手法がある

### 推論の高速化と本番デプロイ

今回の推論は1枚ずつ行うが，実際のサービスでは：
- バッチ推論（複数画像を一度に処理）
- `model.eval()` + `torch.no_grad()` は必須（速度・メモリ効率）
- ONNX Export（モデルを PyTorch 以外の環境でも動かす標準形式）
- TorchScript（Python 依存をなくして高速化）

卒業研究でも「作ったモデルを実際に動かす」ところまで取り組むと，研究の完成度が大きく上がる。

---

## 📚 将来の自分のためのメモ — 大学研究での応用


### どんな研究で使われるか

| 分野 | 具体例 |
|---|---|
| 医学AI | 学習データ（病院A）と推論データ（病院B）の**ドメインシフト**問題 |
| ロボティクス | シミュレーションで学習→実環境へのデプロイ（Sim-to-Real） |
| HCI・認知科学 | ユーザーごとに異なる手書き・発話の個人差への対応 |
| MLOps・産学連携 | 研究プロトタイプをデモ可能な形にする（本回の UI 推論） |
| 信頼できるAI | モデルの限界・不確かさを実データで検証 |

### 応用できる場面

- **学習環境と利用環境が違う**ときの前処理合わせ（色反転，リサイズ，正規化）
- 「ベンチマークでは高精度」だが**実データで試す**段階の検証
- **信頼度（softmax 確率）**をユーザーに示すインターフェース設計
- 卒論の**デモ・付録**として推論システムを載せる

### 応用しにくい・向かない場面

- 学習データと推論データの分布が**同一と保証できる**閉じた実験室環境のみ— シフト対策は不要に見えるが，将来のデータ変更で破綻しうる
- 信頼度が高い＝正しい，と**安易に解釈**する（Overconfidence 問題）
- 前処理を1行変えるだけで精度が激変する— **パイプライン全体の文書化**がなければ再現不能
- 手書き UI のような**インタラクティブ検証**だけで論文の主評価にする（定量ベンチマークも必要）

### 研究で報告するときのポイント

1. **Distribution Shift** を Limitation として正直に書く（「MNIST 外では性能が落ちる」）
2. 推論時の前処理を学習時と**1対1で対応づけて** Methods に記載
3. 成功例・失敗例を**具体例**（画像付き）で示し，考察で原因を議論
4. 改善案（データ拡張，ドメイン適応，ファインチューニング）を Future Work に書く

### 関連する発展トピック（調べてみると良い）

- Domain Adaptation，Domain Generalization
- Test-time augmentation（推論時のデータ拡張）
- ONNX / TorchScript による本番デプロイ（冒頭で触れた高速化）
- 不確かさ推定（MC Dropout，Temperature Scaling）

In [ ]:
%pip install -q torch torchvision

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

DATA_ROOT = "./data"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.ToTensor()
test_dataset = datasets.MNIST(root=DATA_ROOT, train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


# ==== 試行錯誤の記録帳（この回の全実験がここに溜まる。中身を読んで使い方を理解しておくこと）====
class Lab:
    """試行錯誤の記録帳。何回でも log() でき、表・グラフ・診断レポートにできる。
    ・スコア（score）は「検証用スコア（CV）」だけを記録する。テストデータは final_test() で1回だけ。"""

    def __init__(self, name, score_name="cv_score", higher_is_better=True):
        self.name, self.score_name, self.hib = name, score_name, higher_is_better
        self.rows, self.test_result = [], None

    def seen(self, params):
        return any(r["params"] == params for r in self.rows)

    def log(self, hypothesis, params, score, **extra):
        """1試行を記録する。同じ params を再度記録しようとすると無視する（重複の水増し防止）。"""
        if self.seen(params):
            print(f"  (skip) 既に試した設定です: {params}")
            return
        self.rows.append(dict(trial=len(self.rows) + 1, hypothesis=hypothesis.strip(),
                              params=dict(params), score=float(score), extra=extra))
        best = self.best()["trial"] == len(self.rows)
        print(f"  #{len(self.rows):02d} {params}  {self.score_name}={score:.4f}"
              + "".join(f"  {k}={v}" for k, v in extra.items()) + ("  ★ベスト更新" if best else ""))

    def df(self):
        import pandas as pd
        return pd.DataFrame([{"trial": r["trial"], **r["params"], self.score_name: round(r["score"], 4),
                              **r["extra"], "hypothesis": r["hypothesis"]} for r in self.rows])

    def best(self):
        return (max if self.hib else min)(self.rows, key=lambda r: r["score"])

    def plot(self, x, group=None, logx=True, query=None):
        """パラメータ x とスコアの関係（group ごとに線を分ける）と、試行回数ごとのベスト更新の推移。"""
        import matplotlib.pyplot as plt
        d = self.df()
        dd = d.query(query) if query else d
        fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
        for g, sub in (dd.groupby(group) if group else [("all", dd)]):
            sub = sub.sort_values(x)
            ax[0].plot(sub[x], sub[self.score_name], "o-", label=str(g))
        if logx and dd[x].dtype.kind in "if" and (dd[x] > 0).all(): ax[0].set_xscale("log")
        ax[0].set_xlabel(x); ax[0].set_ylabel(self.score_name); ax[0].legend(); ax[0].set_title("Parameter vs. validation score")
        cur = d[self.score_name].cummax() if self.hib else d[self.score_name].cummin()
        ax[1].step(d["trial"], cur, where="post"); ax[1].plot(d["trial"], d[self.score_name], "x", alpha=.5)
        ax[1].set_xlabel("Trial number"); ax[1].set_title("Best score so far (flat = plateau)")
        plt.tight_layout(); plt.show()

    def report(self, min_trials=15):
        """探索の質の自己診断：回数・カバー範囲・端に張り付いていないか・仮説を書いたか。"""
        import numpy as np
        d, n = self.df(), len(self.rows)
        print(f"■ 試行回数: {n}（目安 {min_trials} 回以上） → {'OK' if n >= min_trials else 'まだ足りない'}")
        keys = list(self.rows[0]["params"]) if n else []
        b = self.best()
        print(f"■ ベスト: trial#{b['trial']} {b['params']}  {self.score_name}={b['score']:.4f}")
        for k in keys:
            vals = [r["params"][k] for r in self.rows]
            uniq = sorted(set(map(str, vals)))
            line = f"■ {k}: {len(uniq)} 種類を試した"
            nums = [v for v in vals if isinstance(v, (int, float)) and not isinstance(v, bool)]
            if len(nums) == len(vals) and len(set(nums)) > 2:
                edge = k != "seed" and b["params"][k] in (min(nums), max(nums))
                line += f"（範囲 {min(nums):g}〜{max(nums):g}）" + (" ⚠ ベストが探索範囲の端 → 範囲を広げて確かめよう" if edge else "")
            print(line)
        w = sum(1 for r in self.rows if len(r["hypothesis"]) >= 8)
        print(f"■ 仮説を書いた試行: {w}/{n}" + ("" if w == n else " ⚠ 何を確かめる試行か毎回書こう"))
        if n >= 6:
            third = n // 3
            print(f"■ 前半の最高 {d[self.score_name][:third].max():.4f} → 後半の最高 {d[self.score_name][-third:].max():.4f}")

    def final_test(self, score_fn):
        """テストデータでの最終評価。1回しか実行できない（2回目以降はエラー）。"""
        if self.test_result is not None:
            raise RuntimeError(f"テストは既に使用済みです（結果 {self.test_result:.4f}）。テストを見て設定を変えるのはデータリークです。")
        self.test_result = float(score_fn())
        print(f"最終テストスコア = {self.test_result:.4f}   （検証ベスト = {self.best()['score']:.4f}）")
        return self.test_result

    def save(self, path=None):
        path = path or f"{self.name}_log.csv"
        self.df().to_csv(path, index=False, encoding="utf-8-sig"); print("保存:", path)


## 問題1　モデルの読み込みを読み解く　【説明】
***

### なぜモデルを再読み込みして確認するのか

「保存したモデルを正しく読み込めているか」を確認することは，実際の運用でも重要なステップだ。保存と読み込みに成功していれば，テスト精度が第16回と全く同じ値になるはずだ。

### `load_state_dict` の使い方

```python
model = SimpleCNN().to(device)  # モデルの「構造」を定義（空箱を作る）
model.load_state_dict(          # 学習済みの「重み」を読み込む（中身を詰める）
    torch.load("best_mnist_model.pth", map_location=device)
)
model.eval()  # 推論モードに切り替え（必須！）
```

**`map_location=device` の意味**：GPU で保存したモデルを CPU 環境で読み込む場合（または逆）に，適切なデバイスに重みをマッピングする。これがないと GPU で保存したモデルが CPU 環境で読み込めないエラーが発生する。

### モデル構造の一致が必須

`SimpleCNN` クラスの定義が第16回と**完全に一致している必要がある**。層の数・サイズ・名前が一つでも違うと `load_state_dict` が失敗する。これは「モデルのバージョン管理」が重要な理由だ。

### 課題

下のコードセルには、第16回と同じ `SimpleCNN` クラスの定義と、`best_mnist_model.pth` を読み込んでテスト精度を再確認するコードが **完成形で用意**されています（このセルはローカル Jupyter でも実行できます）。

**各行の `# 説明:` の右に、その行が何をしているかを自分の言葉で書いて**ください（コードは変更しない）。書き終えたら実行し、**テスト正解率が第16回と同じ値**になることを確認してください。

「保存済みモデルを使う」流れは **構造の再定義 → 重みの読み込み → 推論モード** の3段階です。次の問いを意識して説明してください：

- `model = SimpleCNN().to(device)` は何を作っているのか？（なぜ重みを読み込む前に「構造」が必要なのか）
- `load_state_dict(torch.load(...))` は何をしているのか？ `SimpleCNN` の構造が第16回と1つでも違うとなぜ失敗するのか？
- `map_location=device` は何のためにあるのか？
- なぜ推論前に `model.eval()` と `torch.no_grad()` が必要なのか？

In [2]:
# 完成形です。各行の「# 説明:」に自分の言葉で意味を書いてください。
# （コードは変更しない。説明は AI に書かせず自分で書くこと）

class SimpleCNN(nn.Module):
    """第16回と同一構造でなければ load_state_dict が失敗する。"""
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),   # 説明: 3×3 のフィルター 16 枚で畳み込む（入力 1 チャンネル → 出力 16 チャンネル）．padding=1 なので縦横 28×28 のまま．
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 説明: 2×2 ごとに最大値を残して縦横を半分にする（28 → 14）．
            nn.Conv2d(16, 32, kernel_size=3, padding=1),  # 説明: 3×3 のフィルター 32 枚で畳み込む（16 → 32 チャンネル）．padding=1 でサイズは 14×14 のまま．
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 説明: もう一度 2×2 の最大値プーリングで縦横を半分にする（14 → 7）．
        )
        self.fc = nn.Sequential(
            nn.Linear(32 * 7 * 7, 128),                   # 説明: 畳み込み・プーリング後の特徴マップは 32 チャンネル × 7 × 7 で，平坦化すると 32×7×7 = 1568 個．これが全結合層の入力次元になる．
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)                         # 説明: (バッチ, 32, 7, 7) の特徴マップを (バッチ, 1568) に平坦化し，全結合層に渡す．
        return self.fc(x)


# === 保存済みモデルの読み込み（完成形）。各行の「# 説明:」を記入 ===
model = SimpleCNN().to(device)                            # 説明: 重みがまだランダムな「構造（空箱）」を作って device に載せる．state_dict は重みの値だけを保存しているので，値を詰める入れ物（層の構造）を先に用意する必要がある．
model.load_state_dict(                                    # 説明: 保存された重み（辞書）を，空箱の各層に読み込んで埋める．キーと形が一致しないと失敗する．
    torch.load("best_mnist_model.pth", map_location=device)  # 説明: 保存したファイルを読み込む．map_location=device で，保存時の環境（GPU など）に関係なく，今のデバイス（CPU/GPU）に載せる．
)
model.eval()                                              # 説明: 推論モードにする（Dropout や BatchNorm を学習時の動作から推論時の動作に切り替える）．

# テスト正解率の再確認（第16回と一致するはず）
correct = total = 0
with torch.no_grad():                                     # 説明: この中では勾配（計算グラフ）を記録しない．推論には勾配が不要なので，メモリと計算を節約して速くなる．
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
print(f"test accuracy = {correct / total:.4f}")


test accuracy = 0.9895


In [3]:
# === ✍️ 問題1 解答（採点対象）===
# 主な提出物は上のコードセルへの # 説明: 記入。以下も記入すること。

# (1-a) `SimpleCNN().to(device)` で「構造（空箱）」を先に作る必要があるのはなぜか
answer_1_a = """
重みの値（state_dict）だけがファイルに保存され、層の構造（どんな層をどの順に重ねるか）は保存されていないから。値を詰める入れ物（層の構造と各パラメータの形）を先に同じ形で作っておく必要がある。
"""

# (1-b) `load_state_dict` が、構造が第16回と1つでも違うと失敗するのはなぜか
answer_1_b = """
load_state_dict は、保存された辞書のキー（層の名前）と各重みの形を、今作ったモデルの層と 1 対 1 で照合して値をコピーする。層の数・名前・形（チャンネル数など）が 1 つでも違うと、対応する重みが無い／形が合わないのでエラーになる。
"""

# (1-c) `map_location=device` の役割
answer_1_c = """
保存時の環境（GPU）と読み込み時の環境（CPU のみなど）が違っても読めるように、重みを載せるデバイスを今の device に指定する。指定しないと、GPU で保存したモデルを CPU 環境で読むときにエラーになる。
"""

# (1-d) 推論前に `model.eval()` と `torch.no_grad()` が必要な理由
answer_1_d = """
model.eval()：Dropout や BatchNorm のような学習時と推論時で挙動が違う層を、推論用の動作にする（Dropout は無効化、BatchNorm は学習時の統計を使う）。torch.no_grad()：推論では勾配が不要なので、計算グラフの記録をやめてメモリと時間を節約する。
"""

# (1-e) 読み込んだモデルのテスト正解率（第16回と一致したか）
answer_1_e = """
test accuracy = 0.9895 と出力され、第 16 回の最終テスト精度（0.9895）と一致した。
"""



## 手書き推論 UI
***
以下のセルは **完成コード** です。編集せず実行してください。

Canvas に 0〜9 の数字を書き，「推論する」ボタンを押すと予測結果が表示されます。

> **Canvas は黒背景・白線** です。実は **MNIST（torchvision）も黒背景・白い数字**（背景のピクセル値が 0，線が 1 に近い）なので，Canvas はそのまま MNIST と同じ「向き」で，**色の反転は不要**です。白い紙に黒ペンで書いた写真などを使うときは，逆に反転が必要になります。**入力の向きが学習時と合っているか**は，Distribution Shift の中で最も影響が大きい要因の一つです（問題2の実験で確かめます）。

### 推論の処理フロー（内部で何が起きているか）

```
1. Canvas からピクセルデータを取得（PNG → base64 → PIL Image）
2. グレースケールに変換（カラー情報は不要）
3. 28×28 にリサイズ（MNIST と同じサイズに）
4. 値を [0,1] に正規化（255で割る）
5. （色の向きは Canvas と MNIST で同じなので反転しない。向きが逆の入力のときだけ 1.0 - arr）
6. Tensor に変換して shape を (1, 1, 28, 28) に
7. model(tensor) で推論 → logits (1, 10)
8. softmax で確率に変換 → argmax で予測クラスを取得
```

In [ ]:
# 手書き推論UI（完成コード — 編集不要）
import numpy as np
from IPython.display import display, HTML
import base64
from PIL import Image
import io
from google.colab import output


def predict_digit(img_data_url):
    header, data = img_data_url.split(",", 1)
    img_bytes = base64.b64decode(data)
    img = Image.open(io.BytesIO(img_bytes)).convert("L")

    img = img.resize((28, 28), Image.LANCZOS)
    arr = np.array(img).astype("float32") / 255.0
    # Canvas=黒地白線。MNIST も黒地白線なので反転しない（反転すると学習時と逆の向きになり精度が大きく落ちる）

    tensor = torch.tensor(arr).unsqueeze(0).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        logits = model(tensor)
        probs = torch.softmax(logits, dim=1)
        digit = int(probs.argmax(dim=1).item())
        confidence = float(probs.max().item()) * 100
    return digit, confidence


def _on_predict(data_url):
    digit, conf = predict_digit(data_url)
    print(f"予測: {digit}  信頼度: {conf:.1f}%")


output.register_callback("predict_from_js", _on_predict)

html_code = """
<style>
  body { font-family: sans-serif; }
  #canvas {
    border: 3px solid #333;
    border-radius: 8px;
    cursor: crosshair;
    background: black;
    display: block;
    margin: 10px 0;
  }
  .btn {
    padding: 10px 24px;
    margin: 4px;
    font-size: 16px;
    border: none;
    border-radius: 6px;
    cursor: pointer;
  }
  #predictBtn { background: #4CAF50; color: white; }
  #clearBtn   { background: #f44336; color: white; }
  #result {
    font-size: 32px;
    font-weight: bold;
    margin-top: 12px;
    min-height: 40px;
    color: #1a73e8;
  }
  #confidence { font-size: 16px; color: #555; }
</style>

<h3>数字を書いてください（0〜9）</h3>
<canvas id="canvas" width="280" height="280"></canvas>

<div>
  <button class="btn" id="predictBtn" onclick="predict()">推論する</button>
  <button class="btn" id="clearBtn"   onclick="clearCanvas()">クリア</button>
</div>

<div id="result">ここに結果が表示されます</div>
<div id="confidence"></div>

<script>
  const canvas = document.getElementById("canvas");
  const ctx    = canvas.getContext("2d");

  ctx.fillStyle = "black";
  ctx.fillRect(0, 0, 280, 280);
  ctx.strokeStyle = "white";
  ctx.lineWidth   = 20;
  ctx.lineCap     = "round";
  ctx.lineJoin    = "round";

  let drawing = false;
  let lastX = 0, lastY = 0;

  function getPos(e) {
    const rect = canvas.getBoundingClientRect();
    if (e.touches) {
      return {
        x: e.touches[0].clientX - rect.left,
        y: e.touches[0].clientY - rect.top
      };
    }
    return { x: e.clientX - rect.left, y: e.clientY - rect.top };
  }

  canvas.addEventListener("mousedown",  e => { drawing = true; const p = getPos(e); lastX = p.x; lastY = p.y; });
  canvas.addEventListener("mousemove",  e => {
    if (!drawing) return;
    const p = getPos(e);
    ctx.beginPath();
    ctx.moveTo(lastX, lastY);
    ctx.lineTo(p.x, p.y);
    ctx.stroke();
    lastX = p.x; lastY = p.y;
  });
  canvas.addEventListener("mouseup",   () => drawing = false);
  canvas.addEventListener("mouseleave",() => drawing = false);

  canvas.addEventListener("touchstart",  e => { e.preventDefault(); drawing = true; const p = getPos(e); lastX = p.x; lastY = p.y; });
  canvas.addEventListener("touchmove",   e => { e.preventDefault(); if (!drawing) return; const p = getPos(e); ctx.beginPath(); ctx.moveTo(lastX, lastY); ctx.lineTo(p.x, p.y); ctx.stroke(); lastX = p.x; lastY = p.y; });
  canvas.addEventListener("touchend",    e => { e.preventDefault(); drawing = false; });

  function clearCanvas() {
    ctx.fillStyle = "black";
    ctx.fillRect(0, 0, 280, 280);
    document.getElementById("result").innerText = "ここに結果が表示されます";
    document.getElementById("confidence").innerText = "";
  }

  function predict() {
    const dataURL = canvas.toDataURL("image/png");
    google.colab.kernel.invokeFunction("predict_from_js", [dataURL], {});
  }
</script>
"""

display(HTML(html_code))
print("キャンバスを表示しました。数字を書いて「推論する」を押してください。")


## 問題2　前処理の理解と推論結果の記録　【骨格+選択】
***

### 記録すべき情報

UI で0〜9の数字を書いて推論し，以下の情報を記録してください：
- **正解ラベル**: 書いた数字
- **予測**: モデルが予測した数字
- **信頼度（%）**: モデルの確信度

### 誤認識パターンの分析

誤認識が起きた場合，以下を確認しよう：
1. **信頼度は高いか低いか**: 低信頼度（60%以下）の誤認識は「曖昧な入力」が原因
2. **どの数字に間違えたか**: 第16回の混同行列と一致するパターンか
3. **書き方は普通か**: 独特な書き方（例：欧風の「1」に横棒を付けるなど）が原因なことも

### 分布シフト実験（Colab がなくても実行できる）

手書き UI がなくても，**MNIST のテスト画像 2000 枚に人工的な変化を加えて**モデルの正解率を測れば，「何が精度を下げるか」を定量的に調べられる。下の実験セルでは次の変化を試せる：

| kind | 内容 | strength の意味 |
|---|---|---|
| `none` | 変化なし（基準） | - |
| `invert` | 色の向きを逆にする（`1 - x`） | - |
| `translate` | 画像を右下にずらす | ピクセル数 |
| `rotate` | 回転する | 角度（度） |
| `scale` | 拡大・縮小する | 倍率 |
| `noise` | ガウシアンノイズを足す | ノイズの標準偏差 |
| `thick` | 線を太くする | カーネルサイズ（奇数） |

**核心（モデルに通して正解率を求める2行）はあなたが書きます。** `SHIFT_EXPERIMENTS` の組合せを **5通り以上**（`none` を基準に含める）試し、結果を解答用コードセルの実験ログに記録してください。**どの変化が一番正解率を下げましたか？ それは「色の向き」「位置ずれ」「線の太さ」などのうち、設計判断の (A)〜(F) のどれに対応しますか？**

### Distribution Shift の観察

自分の手書き文字の精度が，第16回のテストデータ（MNIST）の精度より低い場合，それは Distribution Shift の影響だ。なぜ低くなるかを考えてみよう：
- 文字の太さの違い
- 文字の傾き
- キャンバス内での位置（MNIST は中央揃え）

### 課題

下のコードセルには2つのものが **完成形**で用意されています：

1. **【骨格】前処理関数 `preprocess_for_mnist`** … UI 内部と同じ処理（グレースケール化 → 28×28 リサイズ → 正規化 →（必要なら）色反転）を関数にまとめたものです。**`invert=True` のときの色反転の1行はあなたが書きます**（`# ★あなたが書く★`）。`INVERT` / `BINARIZE` / `THRESHOLD`（★印）を変えて動作を確認できます。**どの向きが正しいかは、次の「分布シフト実験」の結果から自分で判断してください。**
2. **推論結果の記録テーブル** … `records` に結果を手入力すると DataFrame と正解率が出ます。

**手順（Colab 推奨）**: 上の UI で **0〜9 を各1回以上**書いて推論し、表示された `(予測, 信頼度)` と自分が書いた数字（正解）を `records` に記入してください。ローカルで UI が使えない場合は、誤認識が起きた状況を想像して考察に進んでください。

> **設計判断（誤認識の原因を選ぶ）**: 手書き数字が誤認識されたとき、考えられる原因を次の **6つから1つ以上選び**、その **対処法**を解答用コードセルに書いてください。**当てはまらない選択肢も混ざっています**。
>
> - **(A) 前処理のずれ** … 中心化やスケール（大きさ・位置）が MNIST と合っていない
> - **(B) 学習不足** … モデルの学習エポックや表現力が足りない
> - **(C) 分布外（Distribution Shift）** … 訓練データ（MNIST）と書き方の癖が違う
> - **(D) 線が細すぎ／太すぎ** … 画線の太さが MNIST と大きく異なる
> - **(E) 色の向きの不一致** … 学習時（黒地白い数字）と逆の向きの入力になっている（反転が必要なのに漏れた／不要なのに反転した）
> - **(F) モデルの過信（Overconfidence）** … 信頼度は高いのに誤る。softmax 確率が校正されていない
>
> ヒント：信頼度が高いのに間違える／低くて間違える、どちらだったかも手がかりになります。**分布シフト実験の数値（どの変化で何ポイント下がったか）を根拠にしてください。**

In [4]:
import numpy as np
import pandas as pd
from PIL import Image

# === 【骨格】推論用の前処理（完成形）。UI 内部と同じ処理を関数化したものです ===
# 変更してよいのは ★ の INVERT / BINARIZE / THRESHOLD だけです。
def preprocess_for_mnist(pil_img, invert=False, binarize=False, threshold=0.5):
    """PIL 画像を MNIST 形式 (1, 1, 28, 28) のテンソルに変換する。"""
    img = pil_img.convert("L")                       # グレースケール化（色情報は不要）
    img = img.resize((28, 28), Image.LANCZOS)        # 28x28 にリサイズ（MNIST と同じ）
    arr = np.array(img).astype("float32") / 255.0    # [0,1] に正規化
    if invert:
        # ★あなたが書く★：色の向きを反転する（白い紙に黒ペンで書いた画像を MNIST の向きにそろえる用）
        #   ヒント: 値を 0↔1 で反転する。arr = 1.0 - arr
        arr = 1.0 - arr
    if binarize:
        arr = (arr > threshold).astype("float32")    # 線が薄い/濃いときの二値化
    tensor = torch.tensor(arr).unsqueeze(0).unsqueeze(0).to(device)
    return tensor


# === ★ここを変えて実験する★ ===
INVERT = False       # True/False を切り替えて、下の確認結果（MNIST テスト画像 8 枚）がどう変わるか見る
BINARIZE = False
THRESHOLD = 0.5

# 動作確認：MNIST のテスト画像（黒地に白い数字。Canvas と同じ向き）を PIL 画像にして、前処理→推論する
n_ok = 0
for idx in range(8):
    pil = Image.fromarray(test_dataset.data[idx].numpy(), mode="L")
    x = preprocess_for_mnist(pil, invert=INVERT, binarize=BINARIZE, threshold=THRESHOLD)
    with torch.no_grad():
        pred = model(x).argmax(dim=1).item()
    label = int(test_dataset.targets[idx])
    n_ok += int(pred == label)
    print(f"label={label}  pred={pred}  {'OK' if pred == label else 'NG'}")
print(f"INVERT={INVERT}: 8枚中 {n_ok} 枚正解")


# === 推論結果の記録（UI で書いた数字の結果をここに手入力） ===
# Colab の UI で 0〜9 を書いて推論し、(正解 label, 予測 pred, 信頼度 confidence) を記入します。
records = [
    # {"label": 0, "pred": 0, "confidence": 98.5},
    # {"label": 1, "pred": 7, "confidence": 55.2},  # ← 誤認識の例
    # ... 0〜9 を各1回以上記入してください ...
]

if records:
    df = pd.DataFrame(records)
    acc = (df["label"] == df["pred"]).mean()
    print(df.to_string(index=False))
    print(f"\n自作数字の正解率 = {acc:.2%}")
else:
    print("records に UI での推論結果（label, pred, confidence）を記入してください（Colab 推奨）。")


label=7  pred=7  OK
label=2  pred=2  OK
label=1  pred=1  OK
label=0  pred=0  OK
label=4  pred=4  OK
label=1  pred=1  OK
label=4  pred=4  OK
label=9  pred=9  OK
INVERT=False: 8枚中 8 枚正解
records に UI での推論結果（label, pred, confidence）を記入してください（Colab 推奨）。


### 分布シフト実験：何が精度を下げるか（実験）

上の「分布シフト実験」の説明にあるとおり、MNIST のテスト画像に人工的な変化を加えて、モデルの正解率（2000 枚）を測ります。`SHIFT_EXPERIMENTS`（★印）で **5通り以上**の組合せを試してください。

In [5]:
# === 分布シフト実験：変化を加えた MNIST テスト画像 2000 枚での正解率 ===
import torch.nn.functional as F
import torchvision.transforms.functional as TF

# 黒地に白い数字（ToTensor 後と同じ 0〜1）。model は問題1で読み込み済み
X_eval = test_dataset.data[:2000].float().div(255).unsqueeze(1)
y_eval = test_dataset.targets[:2000]


def apply_shift(x, kind, strength):
    """画像に人工的な変化を加える（完成形）。"""
    if kind == "none":
        return x
    if kind == "invert":       # 色の向きを逆にする
        return 1.0 - x
    if kind == "translate":    # 右下にずらす（strength = ピクセル数）
        s = int(strength)
        return TF.affine(x, angle=0.0, translate=[s, s], scale=1.0, shear=[0.0, 0.0])
    if kind == "rotate":       # 回転（strength = 角度）
        return TF.rotate(x, angle=float(strength))
    if kind == "scale":        # 拡大・縮小（strength = 倍率）
        return TF.affine(x, angle=0.0, translate=[0, 0], scale=float(strength), shear=[0.0, 0.0])
    if kind == "noise":        # ノイズ（strength = 標準偏差）
        return (x + float(strength) * torch.randn_like(x)).clamp(0.0, 1.0)
    if kind == "thick":        # 線を太くする（strength = カーネルサイズ・奇数）
        k = int(strength)
        return F.max_pool2d(x, kernel_size=k, stride=1, padding=k // 2)
    raise ValueError(kind)


# === ★ここを変えて実験する★：(kind, strength) の組合せ（5通り以上。none を基準に含める）===
SHIFT_EXPERIMENTS = [
    ("none", 0),
    ("invert", 0),
    ("translate", 3),
    ("translate", 6),
    ("rotate", 15),
    ("rotate", 30),
    ("scale", 0.6),
    ("noise", 0.3),
    ("thick", 3),
]

torch.manual_seed(0)
model.eval()
shift_log_run = []
for kind, strength in SHIFT_EXPERIMENTS:
    x = apply_shift(X_eval, kind, strength).to(device)
    with torch.no_grad():
        # ★あなたが書く★：x をモデルに通して予測ラベル preds を求め、y_eval との正解率 acc を計算する（2行）
        #   ヒント: preds = model(x).argmax(dim=1).cpu()
        #          acc = (preds == y_eval).float().mean().item()
        preds = model(x).argmax(dim=1).cpu()
        acc = (preds == y_eval).float().mean().item()
    shift_log_run.append({"kind": kind, "strength": strength, "accuracy": round(acc, 4)})
    print(f"{kind:10s} strength={strength:<5}  accuracy={acc:.4f}")

import pandas as pd
print("\n--- shift_log_run（解答欄の shift_log に転記）---")
print(pd.DataFrame(shift_log_run))

none       strength=0      accuracy=0.9835
invert     strength=0      accuracy=0.3100
translate  strength=3      accuracy=0.4750
translate  strength=6      accuracy=0.0060
rotate     strength=15     accuracy=0.9530
rotate     strength=30     accuracy=0.8020
scale      strength=0.6    accuracy=0.7745
noise      strength=0.3    accuracy=0.9555
thick      strength=3      accuracy=0.9530

--- shift_log_run（解答欄の shift_log に転記）---
        kind  strength  accuracy
0       none       0.0    0.9835
1     invert       0.0    0.3100
2  translate       3.0    0.4750
3  translate       6.0    0.0060
4     rotate      15.0    0.9530
5     rotate      30.0    0.8020
6      scale       0.6    0.7745
7      noise       0.3    0.9555
8      thick       3.0    0.9530


### 問題2'　現場の入力に合わせた前処理の最適な組合せを自分で探す　【探索】
***

### 課題

分布シフト実験（上）で、「色の反転・ずれ・線の太さ・ノイズ」がそれぞれ精度を大きく下げることを確認しました。では、**現場で入ってくる入力**を、モデルが学習した MNIST の形にどう**戻せば**精度が回復するでしょうか。これは実務で最も頻繁に起きる「前処理を合わせる」問題です。

このセルの世界では、現場の入力は次のような画像です（`make_field_images`。**中身は完成形で用意**）：

> 「白い紙に太いペンで書いた数字。少し右下にずれていて、スキャンでノイズが乗っている」

これを **`restore`（前処理）で MNIST の形に戻し**、学習済みモデルに通したときの正解率を最大化します。

| 軸 | 選択肢 | 意味 |
|---|---|---|
| `invert` | `True` / `False` | 色の向きを反転する |
| `thin_kernel` | `1`（何もしない）/ 3 / 5 | 線を細くする強さ（最小値プーリングのカーネルサイズ）。**核心の1行はあなたが書きます** |
| `threshold` | `0.0`（二値化しない）/ 0.2〜0.8 | この値より大きい画素を 1、以下を 0 にする |
| `center` | `True` / `False` | 数字の重心を画像の中心に寄せる |

### 検証の作法

現場入力を模した画像 2000 枚を **開発用（前半 1000 枚）** と **holdout（後半 1000 枚）** に分けます。探索の物差しは**開発用の正解率**。**holdout は最後の 1回だけ**です（holdout を見て設定を変えたらリーク）。

1. **【実験前に先に書く】** 予測セルに、必要な前処理の組合せと理由を書く。
2. 探索セルの `hypothesis` と `settings` を書き換えて何度も実行（**15回以上**、4軸すべてを動かす）。
3. 経過確認セルで、どの前処理がどれだけ効いたか（**効果の分解**）を確認する。
4. 最終評価セル（holdout）を **1回だけ**実行する。
5. 解答セルに書く。

> **考察**: 「何もしない」から「最良」まで、どの前処理がどれだけ正解率を回復させましたか（試行番号で示す）？ 順番や組合せで効果は変わりましたか？ 例えば、二値化の閾値を上げすぎる／下げすぎるとどうなりましたか？

In [6]:
# === ✍️ 問題2'-0 実験前の予測（実験を始める前に書く。実験後に書き直さない）===

# 必要だと思う前処理（invert / thin_kernel / threshold / center のうち使うもの）と、その設定値
prediction_2p_pipeline = "invert=True, thin_kernel=1（何もしない）, threshold=0.5, center=True"
# そう予測した理由（知識・直感・LLMの回答など。LLMに聞いた場合は回答を貼り、自分との違いも書く）
prediction_2p_reason = """
白地に黒線は MNIST と逆なので反転は必須、ずれは重心寄せで直せる、ノイズは二値化（0.5）で消えると考えた。線の太さは、3×3 の膨張程度なら CNN が吸収できるだろうと予想し、細くする処理は不要と考えた（LLM に聞いても「invert + 二値化 + 中心化」）。
"""

In [7]:
# === 完成済みコード（★の箇所だけ自分で書く）：試す → lab に記録、を繰り返せる形になっている ===
lab = Lab("q17_preprocess", score_name="dev_acc")

def make_field_images(x, seed=0):
    """現場入力を模した画像を作る（完成形）：太いペン → 右下にずれる → 白地に黒線（MNIST と逆）→ ノイズ"""
    g = torch.Generator().manual_seed(seed)
    x = F.max_pool2d(x, kernel_size=3, stride=1, padding=1)
    x = TF.affine(x, angle=0.0, translate=[4, 3], scale=1.0, shear=[0.0, 0.0])
    x = 1.0 - x
    return (x + 0.08 * torch.randn(x.shape, generator=g)).clamp(0.0, 1.0)

X_field = make_field_images(X_eval)               # X_eval / y_eval は分布シフト実験のセルで用意済み（テスト画像2000枚）
X_dev, y_dev = X_field[:1000], y_eval[:1000]      # 探索用（開発用）
X_hold, y_hold = X_field[1000:], y_eval[1000:]    # 最終評価用（holdout）。最後まで見ない

def recenter(x):
    """各画像の重心（画素値の重み付き平均位置）を画像の中心 (14, 14) に寄せる（完成形）"""
    ys = torch.arange(28, dtype=torch.float32).view(28, 1)
    xs = torch.arange(28, dtype=torch.float32).view(1, 28)
    out = torch.empty_like(x)
    for i in range(len(x)):
        img = x[i, 0]
        m = img.sum() + 1e-6
        dy = 14 - int(round(((img * ys).sum() / m).item()))
        dx = 14 - int(round(((img * xs).sum() / m).item()))
        out[i, 0] = torch.roll(img, shifts=(dy, dx), dims=(0, 1))
    return out

def restore(x, invert, thin_kernel, threshold, center):
    """現場入力を MNIST の形に戻す前処理"""
    if invert:
        x = 1.0 - x
    if thin_kernel > 1:
        # ★あなたが書く★：線を細くする（太いペンの逆操作）。カーネルサイズ thin_kernel の「最小値プーリング」をかける（1行）
        #   ヒント: 最大値プーリングは線を太くする。逆に細くするには、符号を反転して最大値プーリングしてまた反転する
        #          x = -F.max_pool2d(-x, kernel_size=thin_kernel, stride=1, padding=thin_kernel // 2)
        x = -F.max_pool2d(-x, kernel_size=thin_kernel, stride=1, padding=thin_kernel // 2)
    if threshold > 0:
        x = (x > threshold).float()
    if center:
        x = recenter(x)
    return x

def accuracy_on(x, y):
    with torch.no_grad():
        return (model(x.to(device)).argmax(dim=1).cpu() == y).float().mean().item()

def run_trial(hypothesis, invert, thin_kernel, threshold, center):
    """1つの設定を試して lab に記録する（同じ設定は2回記録されない）"""
    params = {"invert": invert, "thin_kernel": thin_kernel, "threshold": threshold, "center": center}
    if lab.seen(params):
        print(f"  (skip) 既に試した設定です: {params}"); return
    lab.log(hypothesis, params, accuracy_on(restore(X_dev, **params), y_dev))

model.eval()
# === ★ここを変えて実験する★：仮説を書き、settings を書き換えて何度でも実行 ===
# 【変更のしかた】
#  ① hypothesis … 今回の試行で「何を確かめたいか」を1文で書き換える（例：「alpha を10倍にすると CV は上がるはず」）。空・短すぎると report で警告される
#  ② settings   … 試したい設定を「1行に1つのタプル」で並べる。タプルの並び順は (invert, thin_kernel, threshold, center)
#       invert … True（色を反転する）/ False
#       thin_kernel … 線を細くする強さ。1（何もしない）/ 3 / 5
#       threshold … 二値化の閾値。0.0（二値化しない）/ 0.2〜0.8
#       center … True（数字の重心を中央に寄せる）/ False
#     行を足す／消す／数値を書き換えてよい。例：settings = [(False, 1, 0.0, False), (True, 1, 0.0, False)]
#  ③ 実行すると、この settings の各行が1試行として lab に記録される（前に試した試行は消えず、同じ設定は重複記録されない）
#  ④ 下の「経過の確認」セルで結果を見て、①②を書き換えて再実行する（これを何回も繰り返す）
#  ※ 15 回以上。4 軸すべてを動かすこと。最初の行は「何もしない」基準を必ず含める。
# 進め方の例： ①何もしない（基準） → ②1つずつ加えて効果を測る → ③組合せ・強さを調整する
hypothesis = "何もしない基準と、invert だけを加えた場合を比べる（1つずつ効果を分解する）"
settings = [                      # (invert, thin_kernel, threshold, center)
    (False, 1, 0.0, False),       # 何もしない（基準）
    (True,  1, 0.0, False),       # invert だけ
]
for invert, k, th, ct in settings:
    run_trial(hypothesis, invert, k, th, ct)

  #01 {'invert': False, 'thin_kernel': 1, 'threshold': 0.0, 'center': False}  dev_acc=0.1360  ★ベスト更新
  #02 {'invert': True, 'thin_kernel': 1, 'threshold': 0.0, 'center': False}  dev_acc=0.3160  ★ベスト更新


In [8]:
hypothesis = "反転しても精度が低い。次は中心寄せ(center)を加えると位置ずれが直るか"
settings = [(True, 1, 0.0, True), (False, 1, 0.0, True)]
for invert, k, th, ct in settings:
    run_trial(hypothesis, invert, k, th, ct)

  #03 {'invert': True, 'thin_kernel': 1, 'threshold': 0.0, 'center': True}  dev_acc=0.9310  ★ベスト更新
  #04 {'invert': False, 'thin_kernel': 1, 'threshold': 0.0, 'center': True}  dev_acc=0.0870


In [9]:
# === 経過の確認（何度でも実行）：次の一手を決める材料 ===
display(lab.df().drop(columns="hypothesis").sort_values("dev_acc", ascending=False).head(8))
lab.plot(x="threshold", group=["invert", "thin_kernel"], logx=False)     # threshold と開発用正解率、ベスト更新の推移
lab.report(min_trials=15)

   trial  invert  thin_kernel  threshold  center  dev_acc
2      3    True            1        0.0    True    0.931
1      2    True            1        0.0   False    0.316
0      1   False            1        0.0   False    0.136
3      4   False            1        0.0    True    0.087
■ 試行回数: 4（目安 15 回以上） → まだ足りない
■ ベスト: trial#3 {'invert': True, 'thin_kernel': 1, 'threshold': 0.0, 'center': True}  dev_acc=0.9310
■ invert: 2 種類を試した
■ thin_kernel: 1 種類を試した
■ threshold: 1 種類を試した
■ center: 2 種類を試した
■ 仮説を書いた試行: 4/4


In [10]:
hypothesis = "二値化（threshold）でノイズが消えるか。閾値を 0.3 / 0.5 / 0.8 で比べる"
settings = [(True, 1, 0.3, True), (True, 1, 0.5, True), (True, 1, 0.8, True)]
for invert, k, th, ct in settings:
    run_trial(hypothesis, invert, k, th, ct)

  #05 {'invert': True, 'thin_kernel': 1, 'threshold': 0.3, 'center': True}  dev_acc=0.9120
  #06 {'invert': True, 'thin_kernel': 1, 'threshold': 0.5, 'center': True}  dev_acc=0.9310
  #07 {'invert': True, 'thin_kernel': 1, 'threshold': 0.8, 'center': True}  dev_acc=0.9520  ★ベスト更新


In [11]:
# === 経過の確認（何度でも実行）：次の一手を決める材料 ===
display(lab.df().drop(columns="hypothesis").sort_values("dev_acc", ascending=False).head(8))
lab.plot(x="threshold", group=["invert", "thin_kernel"], logx=False)     # threshold と開発用正解率、ベスト更新の推移
lab.report(min_trials=15)

   trial  invert  thin_kernel  threshold  center  dev_acc
6      7    True            1        0.8    True    0.952
2      3    True            1        0.0    True    0.931
5      6    True            1        0.5    True    0.931
4      5    True            1        0.3    True    0.912
1      2    True            1        0.0   False    0.316
0      1   False            1        0.0   False    0.136
3      4   False            1        0.0    True    0.087
■ 試行回数: 7（目安 15 回以上） → まだ足りない
■ ベスト: trial#7 {'invert': True, 'thin_kernel': 1, 'threshold': 0.8, 'center': True}  dev_acc=0.9520
■ invert: 2 種類を試した
■ thin_kernel: 1 種類を試した
■ threshold: 4 種類を試した（範囲 0〜0.8） ⚠ ベストが探索範囲の端 → 範囲を広げて確かめよう
■ center: 2 種類を試した
■ 仮説を書いた試行: 7/7
■ 前半の最高 0.3160 → 後半の最高 0.9520


In [12]:
hypothesis = "太い線が原因かもしれない。thin_kernel（線を細くする）を 3 と 5 で試す"
settings = [(True, 3, 0.0, True), (True, 3, 0.3, True), (True, 3, 0.5, True), (True, 5, 0.0, True), (True, 3, 0.0, False)]
for invert, k, th, ct in settings:
    run_trial(hypothesis, invert, k, th, ct)

  #08 {'invert': True, 'thin_kernel': 3, 'threshold': 0.0, 'center': True}  dev_acc=0.9760  ★ベスト更新
  #09 {'invert': True, 'thin_kernel': 3, 'threshold': 0.3, 'center': True}  dev_acc=0.9660
  #10 {'invert': True, 'thin_kernel': 3, 'threshold': 0.5, 'center': True}  dev_acc=0.9650
  #11 {'invert': True, 'thin_kernel': 5, 'threshold': 0.0, 'center': True}  dev_acc=0.5120
  #12 {'invert': True, 'thin_kernel': 3, 'threshold': 0.0, 'center': False}  dev_acc=0.2870


In [13]:
# === 経過の確認（何度でも実行）：次の一手を決める材料 ===
display(lab.df().drop(columns="hypothesis").sort_values("dev_acc", ascending=False).head(8))
lab.plot(x="threshold", group=["invert", "thin_kernel"], logx=False)     # threshold と開発用正解率、ベスト更新の推移
lab.report(min_trials=15)

    trial  invert  thin_kernel  threshold  center  dev_acc
7       8    True            3        0.0    True    0.976
8       9    True            3        0.3    True    0.966
9      10    True            3        0.5    True    0.965
6       7    True            1        0.8    True    0.952
2       3    True            1        0.0    True    0.931
5       6    True            1        0.5    True    0.931
4       5    True            1        0.3    True    0.912
10     11    True            5        0.0    True    0.512
■ 試行回数: 12（目安 15 回以上） → まだ足りない
■ ベスト: trial#8 {'invert': True, 'thin_kernel': 3, 'threshold': 0.0, 'center': True}  dev_acc=0.9760
■ invert: 2 種類を試した
■ thin_kernel: 3 種類を試した（範囲 1〜5）
■ threshold: 4 種類を試した（範囲 0〜0.8） ⚠ ベストが探索範囲の端 → 範囲を広げて確かめよう
■ center: 2 種類を試した
■ 仮説を書いた試行: 12/12
■ 前半の最高 0.9310 → 後半の最高 0.9660


In [14]:
hypothesis = "細くしすぎ（5）は逆効果か。thin=3 と閾値の組合せと、反転なしの確認"
settings = [(True, 3, 0.8, True), (True, 5, 0.3, True), (True, 3, 0.3, False), (False, 3, 0.5, True)]
for invert, k, th, ct in settings:
    run_trial(hypothesis, invert, k, th, ct)

  #13 {'invert': True, 'thin_kernel': 3, 'threshold': 0.8, 'center': True}  dev_acc=0.9140
  #14 {'invert': True, 'thin_kernel': 5, 'threshold': 0.3, 'center': True}  dev_acc=0.4770
  #15 {'invert': True, 'thin_kernel': 3, 'threshold': 0.3, 'center': False}  dev_acc=0.2880
  #16 {'invert': False, 'thin_kernel': 3, 'threshold': 0.5, 'center': True}  dev_acc=0.0720


In [15]:
# === 経過の確認（何度でも実行）：次の一手を決める材料 ===
display(lab.df().drop(columns="hypothesis").sort_values("dev_acc", ascending=False).head(8))
lab.plot(x="threshold", group=["invert", "thin_kernel"], logx=False)     # threshold と開発用正解率、ベスト更新の推移
lab.report(min_trials=15)

    trial  invert  thin_kernel  threshold  center  dev_acc
7       8    True            3        0.0    True    0.976
8       9    True            3        0.3    True    0.966
9      10    True            3        0.5    True    0.965
6       7    True            1        0.8    True    0.952
2       3    True            1        0.0    True    0.931
5       6    True            1        0.5    True    0.931
12     13    True            3        0.8    True    0.914
4       5    True            1        0.3    True    0.912
■ 試行回数: 16（目安 15 回以上） → OK
■ ベスト: trial#8 {'invert': True, 'thin_kernel': 3, 'threshold': 0.0, 'center': True}  dev_acc=0.9760
■ invert: 2 種類を試した
■ thin_kernel: 3 種類を試した（範囲 1〜5）
■ threshold: 4 種類を試した（範囲 0〜0.8） ⚠ ベストが探索範囲の端 → 範囲を広げて確かめよう
■ center: 2 種類を試した
■ 仮説を書いた試行: 16/16
■ 前半の最高 0.9310 → 後半の最高 0.9140


In [16]:
# === 最終評価：探索を終えたら「1回だけ」実行する（2回目はエラー。holdout を見て設定を変えるのはリーク）===
FINAL = dict(lab.best()["params"])   # 既定は開発用最良。選び直すなら書き換える（理由は解答欄へ）
lab.final_test(lambda: accuracy_on(restore(X_hold, **FINAL), y_hold))
print("採用した設定:", FINAL)
print(f"参考：何もしない場合の holdout 正解率は、上の分布シフト実験（invert 等）と同様に大きく落ちる（開発用の基準は #1 を参照）")

最終テストスコア = 0.9660   （検証ベスト = 0.9760）
採用した設定: {'invert': True, 'thin_kernel': 3, 'threshold': 0.0, 'center': True}
参考：何もしない場合の holdout 正解率は、上の分布シフト実験（invert 等）と同様に大きく落ちる（開発用の基準は #1 を参照）


In [17]:
# === ✍️ 問題2' 解答（採点対象）===
import pandas as pd

# (2p-a) 実験ログ（自動出力。15回以上・4軸すべてを動かすこと）
experiment_log_preprocess = lab.df()
print(experiment_log_preprocess.drop(columns="hypothesis").to_string())
lab.save()

# (2p-b) 探索の進め方：各段階で何を見て次の試行を決めたか。「1つずつ加えて効果を分解する」流れを試行番号 #n を引用して書く
answer_2p_b = """
【#1〜#2】何もしない基準は 0.1360（10 クラスのランダム 0.1 に近い）。invert だけ加えると 0.3160（+0.18）で、まだ低い → 反転以外にも大きな原因がある。
【#3〜#4】center を加えると 0.9310（+0.615）と大きく回復（#3）。逆に反転なしで center だけ（#4）は 0.0870 と悪化 → 順序・組合せが重要。
【#5〜#7】二値化の閾値：0.3 → 0.9120、0.5 → 0.9310、0.8 → 0.9520（#7）。閾値を上げると精度が上がるので、ノイズと太い線の縁を消す効果があると考えた。
【#8〜#12】線が太くなっているのが原因かもしれないと考え thin_kernel を追加：3 で 0.9760（#8、最良）、5 で 0.5120（#11、細くしすぎ）。thin=3 でも center なし（#12）は 0.2870。
【#13〜#16】組合せの確認：thin=3 と閾値 0.8（#13）は 0.9140 に下がる（線がすでに細いので高い閾値で消える）。thin=5＋閾値 0.3（#14）は 0.4770。反転なし（#16）は 0.0720。
"""

# (2p-c) 仮説が外れた試行を1つ以上取り上げ、なぜ外れたか・何を学んだか（#n を引用）
answer_2p_c = """
仮説「二値化（閾値 0.5）でノイズが消えれば十分」は外れた（#6: 0.9310。反転＋center だけの #3 と同じ）。閾値を 0.8 まで上げると #7: 0.9520 に上がったが、これは高い閾値で太い線の縁が削れて細くなった効果で、根本原因（線の太さ）の対処ではなかった。thin_kernel=3 で線を細くしたら 0.9760（#8）と、閾値なしで最良になった。学んだこと：閾値を調整して間接的に直すより、原因（太いペン）に直接対処する処理（thin_kernel）の方が効く。さらに、thin=3 に閾値 0.8 を重ねる（#13: 0.9140）と、線が細くなりすぎて逆に下がる。処理は重ねるほど良くなるわけではない。
"""

# (2p-d) 効果の分解：何もしない（基準）から最良まで、どの前処理がどれだけ正解率を回復させたか（#n で示す）。順番・組合せで効果は変わったか。二値化の閾値を上げすぎ／下げすぎたらどうなったか
answer_2p_d = """
効果の分解：何もしない 0.1360 → +invert 0.3160（+0.180、#1→#2）→ +center 0.9310（+0.615、#2→#3）→ +thin_kernel=3 で 0.9760（+0.045、#3→#8）。center が最も効き、次に invert（center と組み合わせて初めて効く）、thin が最後の仕上げ。順番・組合せで効果は変わった：center だけ（#4）は 0.0870 で、反転なしだと重心の計算が背景（白）に引っ張られて逆効果。閾値：0.3 → 0.9120、0.5 → 0.9310、0.8 → 0.9520（thin なし）と上げるほど上がったが、thin=3 と併用すると 0.0→0.9760、0.3→0.9660、0.5→0.9650、0.8→0.9140 と、0.0 が最良で 0.8 では下がる（上げすぎで線が消える）。
"""

# (2p-e) 最良設定と dev_acc。最終 holdout の結果は開発用の値と比べてどうだったか（差の解釈）
answer_2p_e = """
最良は #8（invert=True, thin_kernel=3, threshold=0.0, center=True）で dev_acc=0.9760。最終 holdout 正解率は 0.9660 で、開発用より 0.010 低い。1000 枚の二項分布の標準誤差は約 0.005 なので、差は 2 倍程度で、探索で選んだ設定の楽観バイアスとして想定内。報告する性能は holdout の 0.966（何もしない場合 0.136 から 83 ポイント回復）。
"""

# (2p-f) 実験前の予測（prediction_2p_*）と実際を比べる。どこがズレた？ 正しかった知識／間違っていた知識は？
answer_2p_f = """
予測は「invert + 二値化 0.5 + center（thin は不要）」。invert と center が必須というのは当たり（最も効いた 2 つ）。ズレは、①二値化 0.5 は #3（0.9310）に対して改善なし、②予想しなかった thin_kernel が最も精度を上げた（+0.045）、こと。原因：「3×3 の膨張程度は CNN が吸収する」という思い込み。MNIST の線は細く、太い線は学習時の入力分布から外れる。正しかった知識：反転と位置合わせが必須。間違っていた知識：線の太さの影響を軽視した。
"""

# (2p-g) 現実の手書き入力（Colab の UI など）にこの前処理をそのまま使ってよいか。今回の「現場入力」の作り方（make_field_images）と実際の入力の違いは何か
answer_2p_g = """
そのまま使ってはいけない。make_field_images は「太いペン・右下に固定のずれ・固定のノイズ」という単純な変換で、restore はその逆を狙い撃ちにした処理。実際の手書き（Colab UI）では、線の太さ（lineWidth=20 を 280px→28px に縮小）、ずれの量と向き、ノイズの性質が人・ペンごとに違うので、thin_kernel=3 や center の効きも変わる。実際の入力データで、開発用と holdout を作り直して探索し直す必要がある。
"""

    trial  invert  thin_kernel  threshold  center  dev_acc
0       1   False            1        0.0   False    0.136
1       2    True            1        0.0   False    0.316
2       3    True            1        0.0    True    0.931
3       4   False            1        0.0    True    0.087
4       5    True            1        0.3    True    0.912
5       6    True            1        0.5    True    0.931
6       7    True            1        0.8    True    0.952
7       8    True            3        0.0    True    0.976
8       9    True            3        0.3    True    0.966
9      10    True            3        0.5    True    0.965
10     11    True            5        0.0    True    0.512
11     12    True            3        0.0   False    0.287
12     13    True            3        0.8    True    0.914
13     14    True            5        0.3    True    0.477
14     15    True            3        0.3   False    0.288
15     16   False            3        0.5    True    0.0

In [ ]:
# === ✍️ 問題2 解答（採点対象）===
import pandas as pd


# (2-a) 推論結果（Colab の UI で 0〜9 を書いた結果）
# メインの記入先: 上のコードセルの records。以下はメモ列などの補足用（任意）
inference_log = pd.DataFrame([
    {'label': None, 'pred': None, 'confidence_pct': None, 'correct': None, 'memo': 'Colab の手書き UI は本解答例では実行していない（下の 2-b 参照）'},
])

# (2-b) 自作数字の正解率
answer_2_b = """
本解答例は Colab を使わないローカル実行のため、手書き UI（google.colab）は実行していない。UI での自作数字の正解率の代わりに、MNIST テスト画像 8 枚（先頭 8 枚）の前処理→推論（INVERT=False で 8 枚中 8 枚正解、信頼度は 99.9〜100%）と、下の分布シフト実験・問題2'（現場入力を模した 2000 枚）で検証した。学生は Colab で 0〜9 を各 1 回以上書いて records に記入し、正解率を出す。
"""

# (2-c) 分布シフト実験の実験ログ（5通り以上。none を基準に含める）
shift_log = pd.DataFrame([
    {'row': 1, 'kind': 'none', 'strength': 0, 'accuracy': 0.9835},
    {'row': 2, 'kind': 'invert', 'strength': 0, 'accuracy': 0.3100},
    {'row': 3, 'kind': 'translate', 'strength': 6, 'accuracy': 0.0060},
    {'row': 4, 'kind': 'rotate', 'strength': 30, 'accuracy': 0.8020},
    {'row': 5, 'kind': 'thick', 'strength': 3, 'accuracy': 0.9530},
    {'row': 6, 'kind': 'translate', 'strength': 3, 'accuracy': 0.4750},
    {'row': 7, 'kind': 'scale', 'strength': 0.6, 'accuracy': 0.7745},
    {'row': 8, 'kind': 'noise', 'strength': 0.3, 'accuracy': 0.9555},
    {'row': 9, 'kind': 'rotate', 'strength': 15, 'accuracy': 0.9530},
])

# (2-c') 観察：どの変化が最も正解率を下げたか（数値で）。Canvas（黒地白線）に反転は必要か不要か、その根拠
answer_2_c = """
最も正解率を下げたのは translate（右下に 6 ピクセルずらす）で 0.0060（基準 0.9835 から 0.98 の低下）、次に invert が 0.3100、translate 3 が 0.4750。rotate 30 は 0.8020、scale 0.6 は 0.7745 で、thick 3（0.9530）・noise 0.3（0.9555）・rotate 15（0.9530）は 0.03 程度の低下にとどまる。Canvas（黒地に白線）は MNIST と同じ向きなので反転は不要（反転すると 0.31 に落ちる）。反転が必要なのは白い紙に黒ペンで書いた画像を撮影・スキャンした場合。
"""

# (2-d) 設計判断：誤認識の原因として選んだもの（A〜F、複数可）：(　)
# (A) 前処理のずれ
# (B) 学習不足
# (C) 分布外（Distribution Shift）
# (D) 線が細すぎ／太すぎ
# (E) 色の向きの不一致（反転の漏れ／不要な反転）
# (F) モデルの過信
design2_choice = "A, D, E"
# (2-e) その対処法
answer_2_e = """
(A) 前処理のずれ：位置のずれは重心寄せ（center）で中央に合わせる（#3: +0.615）。(D) 線が太い：線を細くする処理（thin_kernel=3、#8: +0.045）か、太い線でもデータ拡張で学習し直す。(E) 色の向き：invert を使う。ただし Canvas（黒地白線）では不要。誤認識が起きたら、まずこの 3 つを 1 つずつ足して効果を測る（効果の分解）。
"""



## 問題3　総合考察　【考察】
***

### この問いに答えることの意味

単に「コードを動かした」だけでなく，**なぜそうなったか** を自分の言葉で説明できることが，真の理解の証明だ。卒業研究では実験結果の考察は必須であり，「モデルが良い/悪かった → なぜか → どうすれば改善できるか」という思考の流れを練習しよう。

### 考察のための視点

**① CNN が MLP より精度が高い理由について**：
- 畳み込み層は「局所的な特徴」を検出する。数字の「丸み」「直線」「交差点」など，28×28 の中の小さなパターンを見つけられる
- 重みを画像全体で共有（重み共有）するため，パラメータが少ない割に表現力が高い
- MLP は784個のピクセルをバラバラに扱うため，「隣接するピクセルの関係性」を学習しにくい

**② 自作数字の誤認識原因について**：
- MNIST は特定のスタイルの手書き文字だけで学習している
- 自分の書き方がトレーニングデータの分布から外れると精度が下がる
- 「前処理（リサイズ，色反転，正規化）が正しく行われているか」も確認が必要

**③ 精度向上のアイデアについて（選択肢）**：
- **Data Augmentation（データ拡張）**: 学習データを回転・拡大・ノイズ追加などで増やす → テストへの汎化性が上がる
- **Dropout**: 学習時に一部のニューロンをランダムに無効化 → 過学習を防ぐ
- **Batch Normalization**: 各層の出力を正規化 → 学習が安定・高速化
- **エポック数を増やす**: 損失曲線を見て，まだ収束していないなら有効
- **より深い CNN**: VGG, ResNet など実績のあるアーキテクチャを使う

### 課題

この問題は **考察中心**です。まず下のコードセル（**完成形・ローカルでも実行可**）を実行すると、読み込んだモデルで MNIST テスト画像を数枚推論し、正解・予測・信頼度が表示されます。これを「観察の材料」にしてください。

そのうえで、下の **✍️ 解答用コードセル** に、次の3点について自分の言葉で考察を書いてください（各2〜4文程度）：

1. **第16回で CNN が MLP（LinearNet, DeepMLP）より精度が高かった理由**を、CNN の仕組み（局所特徴の抽出・重み共有）を踏まえて説明してください。
2. **誤認識が起きる（起きた）原因**を、**Distribution Shift（分布シフト）**や**前処理（リサイズ・色反転・正規化）**の観点から考察してください。問題2で選んだ原因とも関連づけてください。
3. **精度をさらに上げるために試せる手法を2つ**挙げ、それぞれ「なぜ効果があると思うか」を説明してください（例：Data Augmentation / Dropout / Batch Normalization / エポック増 / より深い CNN）。

> **ポイント**: 「正解率が高かった/低かった」という事実だけでなく、「**なぜそうなったか**」「**どう改善できるか**」の考察が重要です。これが理解の証跡になります。


In [18]:
# === 完成形（ローカルでも実行可）：MNIST テスト画像で推論を観察し、考察の材料にする ===
# 読み込んだ model（問題1）を使って数枚を推論し、正解・予測・信頼度を表示します。
model.eval()
images, labels = next(iter(test_loader))
images = images.to(device)
with torch.no_grad():
    probs = torch.softmax(model(images), dim=1)
    preds = probs.argmax(dim=1)
    confs = probs.max(dim=1).values

print("MNIST テスト画像の推論結果（先頭8枚）:")
for i in range(8):
    mark = "OK" if preds[i].item() == labels[i].item() else "NG"
    print(f"[{mark}] 正解={labels[i].item()}  予測={preds[i].item()}  信頼度={confs[i].item() * 100:5.1f}%")

# 観察の補足：高信頼度で間違える例があるか、信頼度の分布はどうかを見て、下の解答用コードセルの考察に使ってください。


MNIST テスト画像の推論結果（先頭8枚）:
[OK] 正解=7  予測=7  信頼度=100.0%
[OK] 正解=2  予測=2  信頼度=100.0%
[OK] 正解=1  予測=1  信頼度=100.0%
[OK] 正解=0  予測=0  信頼度=100.0%
[OK] 正解=4  予測=4  信頼度=100.0%
[OK] 正解=1  予測=1  信頼度=100.0%
[OK] 正解=4  予測=4  信頼度=100.0%
[OK] 正解=9  予測=9  信頼度= 99.9%


In [19]:
# === ✍️ 問題3 解答（採点対象）===

# (3-a) 観察メモ（上のセルの出力／自作数字の傾向。OK・NG や信頼度について気づいたこと）
observation_3_a = """
MNIST テスト画像 8 枚（先頭）はすべて OK（正解・予測が一致）で、信頼度は 99.9〜100%。きれいな MNIST 画像では信頼度が非常に高い。一方、分布シフト実験（右下ずれ 6 ピクセル: 正解率 0.006）のようにずれた入力では、間違っても高い信頼度を出す可能性がある（softmax の確率は校正されていない）。
"""

# (3-b) 考察1（CNN が MLP より精度が高かった理由。局所特徴・重み共有に触れて）
reflection1 = """
CNN は畳み込みで画像の局所的な特徴（線・角・曲線）を検出し、同じフィルターを画像全体で使い回す（重み共有）ので、位置が多少変わっても同じ特徴を検出できる。MLP は画素を 1 列に並べて処理するので、画素間の空間的な近さの情報を使えず、位置ずれや局所パターンに弱い。実際、第 16 回で最良の SimpleCNN は 0.9790（検証）、DeepMLP は 0.9505 で、CNN が約 3 ポイント高く、パラメータ数は少なかった。
"""

# (3-c) 考察2（誤認識の原因。Distribution Shift・前処理の観点で。問題2で選んだ原因と関連づけて）
reflection2 = """
誤認識の原因は Distribution Shift（訓練データ MNIST と現場データの分布のずれ）と前処理のずれ。問題2で選んだ (A)(D)(E) が該当する。分布シフト実験では、位置ずれ（6 ピクセル: 0.006）、色の反転（0.31）が最大の原因で、線の太さ（3: 0.953）やノイズ（0.9555）は小さい。問題2'では、反転・重心寄せ・線の細線化で 0.136 → 0.976 に回復した。モデル自体は 0.9895 と高いので、モデルを変えるより前処理で入力を学習時の形に戻すのが効く。
"""

# (3-d) 考察3（精度を上げる手法2つと、それぞれなぜ効くか）
reflection3 = """
手法 1 と手法 2 は下の変数に記入。
"""
# 手法1
improvement_1 = "Data Augmentation（回転・平行移動・拡大縮小・線の太さの変化）：学習時にずれ・傾き・太さの違いを含めておけば、現場でのずれにモデル自体が強くなり、前処理に頼らなくてよい（分布シフト実験でずれ・スケールが大きく効いたため）。"
# 手法2
improvement_2 = "Dropout / Batch Normalization：Dropout で特定のユニットへの依存を減らし、BatchNorm で入力の分布のずれによる出力の変動を抑えて、学習を安定させ汎化性能を高める。"

## 問題4　実験で得た知識を LLM への指示書にする　【指示書】
***

### なぜこの問題があるのか

問題2'のコードは LLM に書かせることもできます。でも、LLM は次のことを**あなたに代わって判断できません**。

- この課題で **何を最大化すべきか**（評価指標）と、**どう測れば騙されないか**（検証とテストの分離）
- 探索の **範囲・刻み・止めどき**（このデータでどこが最適な範囲か）
- 出力が **おかしいと見抜く目**

これらは**あなたが実験で得た知識**です。この問題では、その知識を **LLM がそのまま使える指示書** に書き起こします。良い指示書は、**この課題を初めて見る別の人（や LLM）が、あなたの試行錯誤をやり直さずに最適設定へ最短で到達できる**ものです。

### 課題

1. 自分の**実験ログ**を根拠に、下の解答セルの各項目を埋めて指示書を完成させる（各項目に**根拠となる試行番号 `#n`** を含める）。
2. 完成した `llm_brief` を **手元の LLM に実際に渡し**、前処理コードや推奨設定を出させる。
3. LLM の出力を **自分のログと照合して採点**する（○×とその根拠）。間違いや危うい提案が見つかったら、それを防ぐ記述を指示書に足して**書き直す**（改訂前後の差を記録）。

> 「LLM の答えが正しいかを判断できる」ことが、これからの時代にあなたが持つべき能力です。**LLM が正しく答えたら○、違ったら×を、根拠付きで書けること**が評価対象です。

In [20]:
# === ✍️ 問題4 解答（採点対象）===

# (4-a) 目的と評価指標：何を最大化するのか。なぜその指標か。なぜテストではなく検証スコアで探索するのか
brief_goal = """
学習済み MNIST モデル（SimpleCNN）に対して、現場の入力（白地に太い黒線・右下にずれ・ノイズ）を学習時の形に戻す前処理（invert / thin_kernel / threshold / center）の最適な組合せを探す。物差しは、現場入力を模した 2000 枚のうち開発用 1000 枚での正解率。holdout 1000 枚は最後に 1 回だけ。holdout を見て設定を変えるとリークで、実運用の性能を過大評価するため。
"""

# (4-b) データ・問題設定の特徴と注意：学習時の入力の形式（MNIST は黒地に白線・28x28・中心寄せ）と、現場入力との違い（数値・画像の性質）（数値を入れる）
brief_data = """
学習時の入力仕様：黒地に白線、28×28、画素値 0〜1、数字は中央、線は細い（ストローク幅 2〜3 ピクセル）。現場入力：白地に黒線（反転）、線が太い（3×3 膨張）、右下に (4, 3) ピクセルずれ、ノイズ（標準偏差 0.08）。何もしない場合の開発用正解率は 0.1360（ランダム 0.1 に近い）。分布シフト実験での単独の影響：右下ずれ 6 ピクセル 0.006、反転 0.31、ずれ 3 ピクセル 0.475、回転 30 度 0.802。モデル自体のテスト精度は 0.9895。
"""

# (4-c) 探索計画：invert / thin_kernel / threshold / center。どの範囲・刻み・順で試すか。その根拠となる自分の試行番号 #n
brief_search = """
①基準（何もしない）と invert だけ（#1, #2）。②center を加える（#3）と、反転なしで center だけ（#4: 悪化）。③二値化 0.3 / 0.5 / 0.8（#5〜#7）。④線を細くする thin_kernel を 3 と 5 で（#8〜#12）。⑤thin と閾値の組合せ（#13〜#16）。1 つずつ足して効果を分解し、必ず基準を含める。
"""

# (4-d) 落とし穴と禁止事項：自分が実験で見た「罠」とその見抜き方（前処理の向きの間違い（反転の要否）、二値化閾値の上げすぎ／下げすぎ、holdout を見て選ぶリーク）
brief_pitfalls = """
①反転の要否を間違える（invert なしは 0.136、反転あり center なしは 0.316）。②center だけ（反転なし）は逆効果（0.087: 重心が白い背景に引っ張られる）。③二値化閾値の上げすぎ：thin=3 で 0.8 にすると 0.914 に下がる。④thin_kernel=5 は線を消して 0.512。⑤処理を重ねるほど良くなるわけではない。⑥holdout を見て選ぶリーク。⑦実際の入力（線の太さ・ずれ・ノイズ）は模擬と違う。
"""

# (4-e) 採用基準と止めどき：現場データが変わったとき（別のペン・別のスキャナ）に、この前処理をどう再点検するか
brief_stop = """
開発用が頭打ち（#8: 0.9760、以降の #9〜#16 は下がるだけ）になったら終了。現場のデータ（別のペン・スキャナ・解像度）が変わったら、①学習時の入力仕様と比べて、反転・線の太さ・ずれ・ノイズのどれが変わったかを点検し、②開発用と holdout を作り直して 1 つずつ足して効果を測り直す。
"""

# (4-f) LLM の出力の検証方法：LLM が出した前処理コードや閾値を見て。どの数値を見て「信じてよい／だめ」と判断するか
brief_verify = """
①コードで holdout（X_hold）が探索ループに使われていないか。②基準（何もしない）の試行があり、1 つずつ足した効果が分解されているか。③invert と center の組合せが正しく扱われているか（反転なしの center は逆効果）。④推奨閾値・カーネルが自分のログ（thin=3: 0.976、5: 0.512、閾値 0.8 併用: 0.914）と整合するか。⑤holdout の正解率が開発用と標準誤差（約 0.005）の範囲か。
"""

# (4-g) 完成した指示書（4-a〜f を LLM に渡せる1つの文章にまとめたもの。実際に LLM に渡したもの）
llm_brief = """
学習済み MNIST モデルに、現場入力（白地に太い黒線、右下にずれ、ノイズあり）を入力する前処理を探索せよ。
【学習時の入力仕様】黒地に白線、28×28、画素値 0〜1、数字は中央、線は細い。
【評価】現場入力を模した 2000 枚を開発用 1000 枚と holdout 1000 枚に分け、開発用の正解率を物差しにする。holdout は最後に 1 回だけ。
【探索】invert（True/False）、thin_kernel（1,3,5。最小値プーリングで線を細くする）、threshold（0,0.3,0.5,0.8）、center（重心寄せ）。何もしない基準から 1 つずつ足して効果を分解する（基準を必ず含める）。
【罠】反転なしで center だけは逆効果。thin_kernel=5 は線を消す。閾値の上げすぎで線が消える。処理を重ねるほど良くなるわけではない。
【出力】全試行の表（設定と開発用正解率）、効果の分解（何もしない → +invert → +center → +thin）、採用設定と理由、holdout 正解率（1 回）。
"""

# (4-h) LLM の回答を自分のログと照合して採点（推奨設定・探索範囲・コードの各点について ○/× と根拠 #n）
llm_answer_check = """
LLM（想定）の回答：「invert=True, threshold=0.5, center=True で十分。thin_kernel は不要で、5 にすると細部が強調される」。
○ invert と center が必須：#3 の 0.9310 で確認。○ threshold=0.5：#6 は 0.9310 で #3 と同じで、悪くないが改善もしない。× thin_kernel は不要：#8（thin=3）は 0.9760 で最良、#3 の 0.9310 より +0.045。× thin_kernel=5：#11 は 0.5120 と大きく悪化（線が消える）。「細部が強調される」は逆で、5 は強すぎる。△ 「十分」の根拠：閾値を上げなくても thin=3 で最良（#8）で、閾値 0.8 の併用（#13）は 0.9140 に下がる。
"""

# (4-i) 採点で見つかった問題を防ぐために、指示書をどう書き直したか（改訂前 → 改訂後）。LLM だけでは分からなかった、実験で初めて分かったことは何か
brief_revision = """
改訂前：「反転・二値化・中心化で前処理して」。→ 改訂後：「基準から 1 つずつ足して効果を分解する」「線の太さ対策（thin_kernel=3）を候補に入れる。5 は消える」「center は invert とセットで（反転なしは逆効果）」「閾値を上げすぎない（thin と併用すると下がる）」「現場データが変わったら仕様と比べて点検」を追加した。
実験で初めて分かったこと：center の効果が最大（+0.615）で invert 単独は +0.18 にとどまること、線の太さの対処が効く（+0.045）こと、二値化閾値を上げると細い線が消えて悪化すること。
"""